# Compare runs

Loads recorded histories and evaluation summaries for selected runs. Missing runs are skipped so newly added models can appear here before their full training artifacts exist.


In [ ]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from src.data.freihand import FreiHand, SPLIT_SEED, SPLIT_VALIDATION_FRACTION
from src.evaluation.comparison import format_comparison_markdown, load_run_summary
from src.evaluation.overlays import prediction_grid
from src.evaluation.training_curves import load_history, plot_training_curves
from src.models.heatmaps import wrap_with_keypoint_decoder


## Pick runs


In [ ]:
REQUESTED_RUNS = ['baseline-model-1', 'baseline-model-2', 'improved-model-1']
METRICS = ('loss', 'mae')

def has_table_artifacts(run):
    return (
        (project_root / 'logs' / run / 'config.json').exists()
        and (project_root / 'artifacts' / run / 'evaluation.json').exists()
    )

RUNS = [run for run in REQUESTED_RUNS if has_table_artifacts(run)]
missing = [run for run in REQUESTED_RUNS if run not in RUNS]
print('Ready runs:', RUNS)
if missing:
    print('Missing config/evaluation artifacts:', missing)
if not RUNS:
    raise RuntimeError('No runs have both config and evaluation artifacts.')


## Training curves


In [ ]:
for run in RUNS:
    history_path = project_root / 'logs' / run / 'history.json'
    if not history_path.exists():
        print(f'Skipping curves for {run}: missing {history_path}')
        continue
    history = load_history(run)
    fig = plot_training_curves(history, metrics=METRICS, suptitle=f'Training curves: {run}')
    plt.show()


## Comparison table


In [ ]:
summaries = [load_run_summary(run) for run in RUNS]
display(Markdown(format_comparison_markdown(summaries)))


## Representative samples


In [ ]:
dataset = FreiHand()
dataset.validate()
_, val_idx = dataset.train_validation_split(
    validation_fraction=SPLIT_VALIDATION_FRACTION,
    seed=SPLIT_SEED,
)
labels = ['best', 'median', 'p90', 'worst']

for run in RUNS:
    checkpoint_path = project_root / 'models' / run / 'best.keras'
    config_path = project_root / 'logs' / run / 'config.json'
    eval_path = project_root / 'artifacts' / run / 'evaluation.json'
    if not checkpoint_path.exists():
        print(f'Skipping overlays for {run}: missing {checkpoint_path}')
        continue

    config = json.loads(config_path.read_text())
    input_size = int(config.get('input_shape', [224])[0])
    evaluation = json.loads(eval_path.read_text())
    representative = evaluation['metrics']['representative_indices']

    sample_ids = np.array([val_idx[representative[label]] for label in labels])
    images, ground_truth = dataset.load_batch(
        sample_ids,
        image_size=(input_size, input_size),
        flatten_keypoints=False,
    )

    model = keras.models.load_model(checkpoint_path)
    eval_model = wrap_with_keypoint_decoder(model, input_size=input_size) if len(model.output_shape) == 4 else model
    predictions = eval_model.predict(images, verbose=0).reshape(-1, 21, 2)

    titles = [f'{label}' for label in labels]
    fig = prediction_grid(
        images,
        ground_truth,
        predictions,
        titles=titles,
        ncols=2,
        suptitle=f'{run}: representative predictions',
    )
    plt.show()
